In [ ]:
INF = float('inf')

IIT_THRESHOLD = 60_000

IIT_BRACKETS = [
    (36_000, 0.03), (144_000, 0.10), (300_000, 0.20), (420_000, 0.25),
    (660_000, 0.30), (960_000, 0.35), (INF, 0.45),
]

BONUS_TABLE = [
    (3_000,  0.03, 0),      (12_000, 0.10, 210),
    (25_000, 0.20, 1_410),  (35_000, 0.25, 2_660),
    (55_000, 0.30, 4_410),  (80_000, 0.35, 7_160),
    (INF,    0.45, 15_160),
]

SUZHOU = {
    2024: {
        'si_upper': 24_396, 'si_lower': 4_879,
        'hf_periods': ((6, 4_494, 33_000), (6, 4_879, 34_700)),
    },
    2025: {
        'si_upper': 24_762, 'si_lower': 4_952,
        'hf_periods': ((6, 4_879, 34_700), (6, 4_952, 38_900)),
    },
}

SPECIAL_MONTHLY = {
    'housing_loan': 1_000, 'rent': 1_100, 'elderly_care': 3_000,
    'children': 2_000, 'infant': 2_000, 'continuing_edu': 400,
}

In [ ]:
import unicodedata

def progressive_tax(taxable, brackets):
    tax, prev = 0.0, 0
    for upper, rate in brackets:
        if taxable <= prev:
            break
        tax += (min(taxable, upper) - prev) * rate
        prev = upper
    return tax

def separate_tax(amount):
    if amount <= 0:
        return 0.0
    monthly = amount / 12
    for upper, rate, deduction in BONUS_TABLE:
        if monthly <= upper:
            return amount * rate - deduction
    return 0.0

def _capped_base(amount, lower, upper):
    if amount <= 0:
        return 0.0
    return min(max(amount, lower), upper)

def take_home_suzhou(monthly_gross, year, bonus=0, equity_vest=0, hf_rate=0.12,
                     housing_loan=False, rent=False, elderly_care=False,
                     children=0, infant=0, continuing_edu=False, *,
                     si_monthly_base=None, hf_monthly_base=None,
                     bonus_tax_mode='auto', equity_tax_mode='qualified'):
    if min(monthly_gross, bonus, equity_vest) < 0:
        raise ValueError('Income cannot be negative')
    if not 0.05 <= hf_rate <= 0.12:
        raise ValueError('hf_rate must be between 5% and 12%')
    if children < 0 or infant < 0:
        raise ValueError('Child deduction counts cannot be negative')
    if bonus_tax_mode not in {'auto', 'separate', 'combined'}:
        raise ValueError("bonus_tax_mode must be 'auto', 'separate', or 'combined'")
    if equity_tax_mode not in {'qualified', 'wages'}:
        raise ValueError("equity_tax_mode must be 'qualified' or 'wages'")

    housing_loan_share = float(housing_loan)
    if housing_loan_share not in {0.0, 0.5, 1.0}:
        raise ValueError('housing_loan must be false, true, or 0.5')
    if housing_loan_share and rent:
        raise ValueError('Housing loan interest and rent cannot both be deducted')

    d = SUZHOU[year]
    si_income = monthly_gross if si_monthly_base is None else si_monthly_base
    hf_income = monthly_gross if hf_monthly_base is None else hf_monthly_base
    si_base = _capped_base(si_income, d['si_lower'], d['si_upper'])
    if 'hf_periods' in d:
        hf_periods = d['hf_periods']
    else:
        hf_periods = ((12, d['hf_lower'], d['hf_upper']),)
    if sum(months for months, _, _ in hf_periods) != 12:
        raise ValueError('Housing fund periods must cover 12 months')

    pension = si_base * 0.08 * 12
    medical = si_base * 0.02 * 12
    unemployment = si_base * 0.005 * 12
    housing_fund = sum(
        _capped_base(hf_income, lower, upper) * hf_rate * months
        for months, lower, upper in hf_periods
    )
    total_pretax = pension + medical + unemployment + housing_fund

    if isinstance(elderly_care, bool):
        elderly_care_monthly = SPECIAL_MONTHLY['elderly_care'] if elderly_care else 0
    else:
        elderly_care_monthly = float(elderly_care)
    if not 0 <= elderly_care_monthly <= SPECIAL_MONTHLY['elderly_care']:
        raise ValueError('elderly_care monthly deduction must be between 0 and 3000')

    special = housing_loan_share * SPECIAL_MONTHLY['housing_loan']
    if rent:           special += SPECIAL_MONTHLY['rent']
    special += elderly_care_monthly
    special += children * SPECIAL_MONTHLY['children']
    special += infant * SPECIAL_MONTHLY['infant']
    if continuing_edu: special += SPECIAL_MONTHLY['continuing_edu']
    annual_special = special * 12

    annual_gross = monthly_gross * 12
    comprehensive_income = annual_gross
    if equity_tax_mode == 'wages':
        comprehensive_income += equity_vest
    base_deductions = total_pretax + annual_special + IIT_THRESHOLD
    separate_taxable = max(0, comprehensive_income - base_deductions)
    combined_taxable = max(0, comprehensive_income + bonus - base_deductions)
    separate_iit = progressive_tax(separate_taxable, IIT_BRACKETS)
    separate_btax = separate_tax(bonus)
    combined_iit = progressive_tax(combined_taxable, IIT_BRACKETS)

    selected_bonus_mode = bonus_tax_mode
    if selected_bonus_mode == 'auto':
        selected_bonus_mode = (
            'combined' if combined_iit < separate_iit + separate_btax else 'separate'
        )
    if selected_bonus_mode == 'combined':
        taxable, iit, btax = combined_taxable, combined_iit, 0.0
    else:
        taxable, iit, btax = separate_taxable, separate_iit, separate_btax

    etax = (
        progressive_tax(equity_vest, IIT_BRACKETS)
        if equity_tax_mode == 'qualified' else 0.0
    )
    total_income = annual_gross + bonus + equity_vest
    total_tax = iit + btax + etax
    net = total_income - total_pretax - total_tax

    return dict(monthly_gross=monthly_gross, annual_gross=annual_gross,
                bonus=bonus, equity_vest=equity_vest,
                pension=pension, medical=medical, unemployment=unemployment,
                housing_fund=housing_fund, total_pretax=total_pretax,
                annual_special=annual_special, taxable=taxable,
                iit=iit, btax=btax, etax=etax, total_tax=total_tax,
                bonus_tax_mode=selected_bonus_mode, equity_tax_mode=equity_tax_mode,
                total_income=total_income, net=net)

def show_cn(r, year):
    g = r['total_income']
    PAD_W = 14
    S = '\u3000'
    def dw(s):
        return sum(2 if unicodedata.east_asian_width(c) in ('W', 'F') else 1 for c in s)
    def pad(s):
        need = PAD_W * 2 - dw(s)
        return s + S * (need // 2) + ' ' * (need % 2)
    def row(label, val, rate=True):
        pct = f"（{val/g*100:4.1f}%）" if rate and g else ""
        print(f"　　{pad(label)}￥{val:>11,.2f}{pct}")
    SEP = '─' * 50
    print(SEP)
    print(f"　　{year} · 苏州 · 月薪 ￥{r['monthly_gross']:,.0f}")
    print(SEP)
    row('年度工资', r['annual_gross'], rate=False)
    if r['bonus']:       row('年终奖', r['bonus'], rate=False)
    if r['equity_vest']: row('股权激励', r['equity_vest'], rate=False)
    row('年度总收入', g, rate=False)
    print()
    row('养老保险', r['pension'])
    row('医疗保险', r['medical'])
    row('失业保险', r['unemployment'])
    row('住房公积金', r['housing_fund'])
    print()
    if r['annual_special']:
        row('专项附加扣除', r['annual_special'], rate=False)
    row('综合所得个税', r['iit'])
    if r['btax']: row('年终奖个税', r['btax'])
    if r['etax']: row('股权激励个税', r['etax'])
    print(SEP)
    row('五险一金合计', r['total_pretax'])
    row('个税合计', r['total_tax'])
    print(SEP)
    row('年度到手', r['net'])
    row('月度到手', r['net'] / 12, rate=False)
    rate = (r['total_pretax'] + r['total_tax']) / g * 100 if g else 0
    print(f"　　{pad('综合扣除率')}{rate:>12.1f}%")
    print(SEP)

In [ ]:
monthly_gross  = 13_000 # income of a typical engineer
bonus          = 20_000
equity_vest    = 0
year           = 2025
hf_rate        = 0.12
housing_loan   = True
rent           = False
elderly_care   = False
children       = 0
infant         = 0
continuing_edu = False

r = take_home_suzhou(
    monthly_gross, year, bonus, equity_vest, hf_rate,
    housing_loan, rent, elderly_care, children, infant, continuing_edu,
    bonus_tax_mode='auto', equity_tax_mode='qualified',
)
show_cn(r, year)

──────────────────────────────────────────────────
　　2025 · 苏州 · 月薪 ￥13,000
──────────────────────────────────────────────────
　　年度工资　　　　　　　　　　￥ 156,000.00
　　年终奖　　　　　　　　　　　￥  20,000.00
　　年度总收入　　　　　　　　　￥ 176,000.00

　　养老保险　　　　　　　　　　￥  12,480.00（ 7.1%）
　　医疗保险　　　　　　　　　　￥   3,120.00（ 1.8%）
　　失业保险　　　　　　　　　　￥     780.00（ 0.4%）
　　住房公积金　　　　　　　　　￥  18,720.00（10.6%）

　　专项附加扣除　　　　　　　　￥  12,000.00
　　工资个税　　　　　　　　　　￥   2,370.00（ 1.3%）
　　年终奖个税　　　　　　　　　￥     600.00（ 0.3%）
──────────────────────────────────────────────────
　　五险一金合计　　　　　　　　￥  35,100.00（19.9%）
　　个税合计　　　　　　　　　　￥   2,970.00（ 1.7%）
──────────────────────────────────────────────────
　　年度到手　　　　　　　　　　￥ 137,930.00（78.4%）
　　月度到手　　　　　　　　　　￥  11,494.17
　　综合扣除率　　　　　　　　　        21.6%
──────────────────────────────────────────────────
